# 🐳 Deploy: From Notebook to Production API

You've built and tested the multi-agent workflow in notebooks. Now ship it.

In this challenge you'll:
1. **Explore** the production FastAPI app that wraps your workflow
2. **Build & run** the Docker container locally
3. **Hit the API** to process all 3 incidents
4. **Generate a report** — a tangible deliverable you can take home

| Step | Time | What You Do |
|------|------|-------------|
| Explore the API code | 2 min | Read `app/main.py` — see your workflow as a REST service |
| Docker build & run | 3 min | `docker compose up --build` |
| Process incidents | 3 min | POST all 3 incidents to the API |
| Generate report | 2 min | Collect results into a formatted summary |

---
## Step 1: Explore the Production App

Your notebook workflow has been packaged into a production-ready FastAPI service.

| File | What It Does |
|------|-------------|
| `app/main.py` | FastAPI server with `/health`, `/incidents`, `/incidents/stream` endpoints |
| `app/workflow.py` | Your Challenge 2 workflow — same agents, same routing, same tools |
| `Dockerfile` | Python 3.12 slim, non-root user, health check |
| `docker-compose.yml` | One command to build + run with your `.env` |

**Key difference from notebooks:** The production app uses `DefaultAzureCredential` instead of `AzureCliCredential` — this works both locally (falls back to CLI) and in the cloud (uses managed identity).

Take a quick look at the API endpoints:

In [ ]:
# Quick look at the API structure
print("📡 API Endpoints:")
print()
print("  GET  /health       → Health check (is the workflow ready?)")
print("  POST /incidents    → Submit an incident, get structured response")
print("  POST /incidents/stream → Submit and stream results as SSE")
print()
print("📁 Production files:")
print("  app/main.py        → FastAPI server")
print("  app/workflow.py    → Your MAF workflow (same as Challenge 2)")
print("  Dockerfile         → Container image definition")
print("  docker-compose.yml → One-command deployment")

---
## Step 2: Build & Run with Docker

Run these commands in your **terminal** (not in this notebook):

```bash
# From the repo root (maf-lab/)
docker compose up --build
```

You should see:
```
maf-incident-api-1  | INFO:     Started server process
maf-incident-api-1  | INFO:     Building MAF incident response workflow...
maf-incident-api-1  | INFO:     Workflow ready.
maf-incident-api-1  | INFO:     Uvicorn running on http://0.0.0.0:8000
```

**Leave it running** and continue below.

> ⚠️ **No Docker?** You can run the API directly: `uvicorn app.main:app --port 8000`

In [ ]:
# Verify the API is running
import httpx

API_URL = "http://localhost:8000"

try:
    r = httpx.get(f"{API_URL}/health", timeout=5)
    health = r.json()
    print(f"✅ API is running!")
    print(f"   Status: {health['status']}")
    print(f"   Version: {health['version']}")
    print(f"   Workflow ready: {health['workflow_ready']}")
except httpx.ConnectError:
    print("❌ API not reachable at localhost:8000")
    print("   Run: docker compose up --build")
    print("   Or:  uvicorn app.main:app --port 8000")

---
## Step 3: Process All 3 Incidents via the API

Now hit your production API with all 3 incidents — the same ones from the notebooks,
but this time going through Docker → FastAPI → MAF Workflow.

In [ ]:
import json
import httpx
from datetime import datetime

API_URL = "http://localhost:8000"

with open("../data/incidents.json") as f:
    incidents = json.load(f)

results = []

for incident in incidents:
    print(f"\n{'='*50}")
    print(f"🚨 Sending: {incident['title']}")
    print(f"   Service: {incident['service']} | Severity: {incident['severity']}")
    
    r = httpx.post(f"{API_URL}/incidents", json=incident, timeout=60)
    
    if r.status_code == 200:
        resp = r.json()
        results.append(resp)
        print(f"   ✅ Processed in {resp['started_at']} → {resp['completed_at']}")
        for output in resp['outputs']:
            print(f"   📋 {output[:120]}..." if len(output) > 120 else f"   📋 {output}")
    else:
        print(f"   ❌ Error {r.status_code}: {r.text}")
        results.append({"error": r.text, "incident": incident['title']})

print(f"\n{'='*50}")
print(f"✅ Processed {len(results)} incidents via the API")

---
## Step 4: Generate Incident Response Report

Create a formatted report summarizing everything the system did.
This is your **deliverable** — proof that you built and deployed a production multi-agent system.

<div style="border: 1px solid #e94560; border-left: 4px solid #e94560; padding: 16px 20px; border-radius: 6px; background-color: rgba(233, 69, 96, 0.08);">

**✍️ Your Turn** — Generate the report

Place your cursor in the cell below and press `Ctrl+I` to let Copilot generate the report from the comments.

</div>

In [ ]:
# Generate a formatted incident response report from the `results` list.
# Each item in results is a dict with: run_id, incident_id, status, outputs, started_at, completed_at
# The report should include:
# 1. A header: "MAF Incident Response Report" with the current date
# 2. Summary stats: total incidents processed, how many completed vs errored
# 3. For each incident:
#    - Incident ID and title (from the outputs or incident data)
#    - Status (completed/error)
#    - Full output text
#    - Time taken (parse started_at and completed_at, compute delta)
# 4. A footer: "Generated by MAF Workshop — WeAreDevelopers 2026"
# Print the report using print() with nice formatting (use = and - for separators)

---
## Step 5 (Bonus): Deploy to Azure Container Apps

If you have time, push your Docker image to Azure and deploy it as a serverless container.

The `infrastructure/` folder has Terraform configs for:
- Azure Container Registry (ACR)
- Azure Container Apps Environment
- Container App with managed identity → Foundry

```bash
# 1. Build & push image to ACR
az acr login --name <your-acr>
docker tag maf-lab-maf-incident-api <your-acr>.azurecr.io/maf-incident-api:latest
docker push <your-acr>.azurecr.io/maf-incident-api:latest

# 2. Deploy with Terraform
cd infrastructure
terraform init
terraform apply

# 3. Test your cloud endpoint
curl https://<your-app>.azurecontainerapps.io/health
```

Your multi-agent system is now running in the cloud. 🚀

---
## 🎉 You Did It!

### What You've Built Today

```
Challenge 0: Setup          → Verified Azure + MAF connection
Challenge 1: Agents         → Typed agents with structured outputs
Challenge 2: Workflows      → DAG orchestration with conditional routing
Challenge 3: HITL           → Human approval gates for safe operations
Challenge 4: Advanced       → Composition, tracing, or parallelism
Deploy:      Production     → Docker container serving a REST API
```

### Take It Home

Fork the repo, swap the mock tools for real ones (Prometheus, K8s, PagerDuty),
and you have a production incident response system.

**Resources:**
- [MAF GitHub](https://github.com/microsoft/agent-framework)
- [Azure AI Foundry](https://ai.azure.com)
- [This workshop repo](https://github.com/ishasalania/maf-lab)